In [1]:
library(data.table)
library(stringr)
library(dplyr)
library(ggplot2)
library(readxl)


Attaching package: 'dplyr'


The following objects are masked from 'package:data.table':

    between, first, last


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




## Create 2024 Global Workbook File

In [2]:
##Collect all 2024 workbook files and give them a unique name based on their county
myfiles24 <- Sys.glob("/anvil/projects/tdm/sps/tippecanoe-project/data/tippecanoe_copy/*2024 WORKBOOK.csv")
fixfilename2024 <- function(x) str_to_title(str_remove(str_remove(x, "/anvil/projects/tdm/sps/tippecanoe-project/data/tippecanoe_copy/"),  " 2024 WORKBOOK[A-Za-z \\.]*"))
names(myfiles24) <- sapply(myfiles24, fixfilename2024)
options(repr.matrix.max.cols=100)

In [3]:
## Create function to store all workbook files in a dataframe separated by County
makecountyDF <- function(x) {
  tempDF <- cbind(names(myfiles24)[x], fread(myfiles24[x]))
  names(tempDF)[1] <- "County"
  return(tempDF)
}


In [4]:
## Store all 2024 workbook files in a dataframe separated by County
myresults24 <- lapply(1:length(myfiles24), makecountyDF)
names(myresults24) <- names(myfiles24)
sapply(myresults24, dim)

Adams,Allen,Bartholomew,Benton,Blackford,Boone,Brown,Carroll,Cass,Clark,Clay,Clinton,Crawford,Daviess,Dearborn,Decatur,Dekalb,Delaware,Dubois,Elkhart,Fayette,Fountain,Franklin,Fulton,Gibson,Grant,Greene,Huntington,Jackson,Jasper,Jay,Jefferson,Jennings,Johnson,Knox,Kosciusko,Lagrange,Laporte,Marshall,Miami,Monroe,Montgomery,Morgan,Newton,Noble,Ohio,Perry,Pike,Porter,Posey,Pulaski,Putnam,Randolph,Ripley,Rush,Scott,Shelby,Spencer,St. Joseph,Starke,Steuben,Sullivan,Switzerland,Tipton,Union,Vanderburgh,Vermillion,Vigo,Wabash,Warrick,Washington,Wayne,Wells,White,Whitley
19632,165863,38466,7803,8742,37535,15738,21380,25224,62209,23762,23154,11203,22288,32294,17782,31403,59291,35166,87544,14094,14512,17652,17216,22085,36849,26670,22760,26586,21404,15639,20252,20555,67433,31050,52177,27565,64656,30306,22757,55314,23453,42091,11338,26837,3520,16415,20182,83338,19693,17941,25828,18464,17849,12049,15294,30102,22066,118678,19773,39195,19415,8357,12152,6433,79307,12620,59828,19432,43006,19485,35109,16412,21634,20335
20,20,18,20,20,19,18,20,20,18,18,20,21,20,19,20,20,20,18,20,20,17,17,20,19,20,17,19,19,17,20,19,19,19,19,20,20,20,20,20,20,20,20,20,20,20,18,20,23,19,18,20,20,20,17,19,20,18,20,17,20,17,20,20,20,20,17,19,20,19,19,17,20,20,18


In [5]:
#For each attribute that will remain in the global file, search for (1) existence and (2) data type in each 
## County's file, and correct any outliers so that the counties can be combined easily
myparcelnumbers <- lapply(1:75, function(x) myresults24[[x]]$"Parcel Number")
which(sapply(myparcelnumbers, is.null))
table(sapply(myparcelnumbers, class))
myprioryearpropertyclass <- lapply(1:75, function(x) myresults24[[x]]$"Prior Year Property Class")
which(sapply(myprioryearpropertyclass, is.null))
table(sapply(myprioryearpropertyclass, class))
mycurrentyearpropertyclass <- lapply(1:75, function(x) myresults24[[x]]$"Current Year Property Class")
which(sapply(mycurrentyearpropertyclass, is.null))
table(sapply(mycurrentyearpropertyclass, class))
myprioryeartotalAV <- lapply(1:75, function(x) as.numeric(gsub(",", "", myresults24[[x]]$"Prior Year Total AV")))
which(sapply(mycurrentyearpropertyclass, is.null))
myresults24[[11]]$"Prior Year Total AV" <- gsub("\\$", "", myresults24[[11]]$"Prior Year Total AV")
myprioryeartotalAV[[11]] <- as.numeric(gsub(",", "", myresults24[[11]]$"Prior Year Total AV"))
which(is.na(myprioryeartotalAV))
table(sapply(myprioryeartotalAV, class))
mycurrentyeartotalAV <- lapply(1:75, function(x) as.numeric(gsub(",", "", myresults24[[x]]$"Current Year Total AV")))
which(sapply(mycurrentyeartotalAV, is.null))
myresults24[[11]]$"Current Year Total AV" <- gsub("\\$", "", myresults24[[11]]$"Current Year Total AV")
mycurrentyeartotalAV[[11]] <- as.numeric(gsub(",", "", myresults24[[11]]$"Current Year Total AV"))
which(is.na(myprioryeartotalAV))
table(sapply(mycurrentyeartotalAV, class))

integer(0)


character 
       75 

integer(0)


integer 
     75 

integer(0)

...

Warning message in FUN(X[[i]], ...):
"NAs introduced by coercion"


integer(0)

Warning message:
"NAs introduced by coercion"


integer(0)


numeric 
     75 

In [6]:
mytownship <- lapply(1:75, function(x) myresults24[[x]]$"Township Name")
which(sapply(mytownship, is.null))
##myparcelnumbers[[33]] <- myresults24[[33]]$"State Parcel Number"
table(sapply(mytownship, class))
mytd <- lapply(1:75, function(x) myresults24[[x]]$"Taxing District")
which(sapply(mytd, is.null))
mytd[[47]] <- myresults24[[47]]$"Tax District"
table(sapply(mytd, class))

integer(0)


character 
       75 

[1] 47


integer 
     75 

In [7]:
#Create the 2024 Global Workbook file by looping over each county and extracting the relevant attributes
combinedwkbk24 = data.frame(
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PriorPropertyClass = integer(),
  CurrentPropertyClass = integer(), 
  PriorAV = numeric(),
  CurrentAV = numeric()
)
for (i in 1:75) {
    newdata = data.frame(
      County = myresults24[[i]]$'County',
      Township = mytownship[[i]],
      TaxDistrict = mytd[[i]],
      ParcelNumber = myparcelnumbers[[i]],
      PriorPropertyClass = myprioryearpropertyclass[[i]],
      CurrentPropertyClass = mycurrentyearpropertyclass[[i]], 
      PriorAV = myprioryeartotalAV[[i]],
      CurrentAV = mycurrentyeartotalAV[[i]]
    )
    combinedwkbk24 = rbind(combinedwkbk24,newdata)
}
head(combinedwkbk24)

,County,Township,TaxDistrict,ParcelNumber,PriorPropertyClass,CurrentPropertyClass,PriorAV,CurrentAV
,<chr>,<chr>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>
1,Adams,PREBLE TOWNSHIP,12,01-01-01-100-001.000-012,101,101,231500,250000
2,Adams,PREBLE TOWNSHIP,12,01-01-01-100-002.000-012,100,100,69700,83700
3,Adams,PREBLE TOWNSHIP,12,01-01-01-100-002.500-012,100,100,92000,110400
4,Adams,PREBLE TOWNSHIP,12,01-01-01-100-004.000-012,685,685,244100,260400
5,Adams,PREBLE TOWNSHIP,12,01-01-01-100-005.000-012,610,610,0,0
6,Adams,PREBLE TOWNSHIP,12,01-01-01-100-006.000-012,610,610,0,0


## Create 2023 Global Workbook File

In [8]:
##Collect all 2023 workbook files and give them a unique name based on their county
myfiles23 <- Sys.glob("/anvil/projects/tdm/sps/tippecanoe-project/data/tippecanoe_copy/*2023 WORKBOOK.csv")
fixfilename2023 <- function(x) str_to_title(str_remove(str_remove(x, "/anvil/projects/tdm/sps/tippecanoe-project/data/tippecanoe_copy/"),  " 2023 WORKBOOK[A-Za-z \\.]*"))
names(myfiles23) <- sapply(myfiles23, fixfilename2023)
options(repr.matrix.max.cols=100)

In [9]:
## Create function to store all workbook files in a dataframe separated by County
makecountyDF <- function(x) {
  tempDF <- cbind(names(myfiles23)[x], fread(myfiles23[x]))
  names(tempDF)[1] <- "County"
  return(tempDF)
}

In [10]:
## Store all 2024 workbook files in a dataframe separated by County
myresults23 <- lapply(1:length(myfiles23), makecountyDF)
names(myresults23) <- names(myfiles23)
sapply(myresults23, dim)

Adams,Allen,Bartholomew,Benton,Blackford,Boone,Brown,Carroll,Cass,Clark,Clay,Clinton,Crawford,Daviess,Dearborn,Decatur,Dekalb,Delaware,Dubois,Elkhart,Fayette,Floyd,Fountain,Franklin,Fulton,Gibson,Grant,Greene,Hamilton,Hancock,Harrison,Hendricks,Henry,Howard,Huntington,Jackson,Jasper,Jay,Jefferson,Jennings,Johnson,Knox,Kosciusko,Lagrange,Lake,Laporte,Lawrence,Madison,Marion,Marshall,Martin,Miami,Monroe,Montgomery,Morgan,Newton,Noble,Ohio,Orange,Owen,Parke,Perry,Pike,Porter,Posey,Pulaski,Putnam,Randolph,Ripley,Rush,Scott,Shelby,Spencer,St. Joseph,Starke,Steuben,Sullivan,Switzerland,Tippecanoe,Tipton,Union,Vanderburgh,Vermillion,Vigo,Wabash,Warren,Warrick,Washington,Wayne,Wells,White
19545,164251,38360,7841,8797,36512,15748,21331,25174,61403,23713,23176,11109,22138,32192,17757,31329,59274,34953,87527,14086,40024,14425,17531,17190,21909,36802,26552,142647,41699,27239,76978,28139,43604,22737,26422,21379,15657,20177,20498,67576,31046,52208,27536,241920,64617,31431,83736,344019,30365,10280,22804,55234,23349,41965,11327,26747,3512,21635,17563,20045,16413,20175,82973,19651,17850,25882,18365,18395,11958,15154,29893,22289,118696,19792,38986,1048575,8311,68389,12286,6404,79365,12608,59798,19352,10346,43022,19419,35067,16259,21598
20,20,20,20,20,20,17,20,20,17,18,20,21,20,20,20,20,20,21,20,20,20,17,17,20,22,20,22,20,20,17,20,14,20,17,20,17,20,20,20,20,22,20,20,15,20,20,20,17,20,20,20,20,20,20,20,20,20,20,22,16,22,20,20,22,20,20,20,20,17,20,20,21,20,20,20,20,20,20,20,20,20,20,18,20,20,22,20,20,20,20


In [11]:
#For each attribute that will remain in the global file, search for (1) existence and (2) data type in each 
## County's file, and correct any outliers so that the counties can be combined easily
myparcelnumbers <- lapply(1:91, function(x) myresults23[[x]]$"Parcel Number")
which(sapply(myparcelnumbers, is.null))
myparcelnumbers[[33]] <- myresults23[[33]]$"State Parcel Number"
table(sapply(myparcelnumbers, class))
myprioryearpropertyclass <- lapply(1:91, function(x) myresults23[[x]]$"Prior Year Property Class")
which(sapply(myprioryearpropertyclass, is.null))
myprioryearpropertyclass[[7]] <- myresults23[[7]]$"Prior  Year Property Class"
myprioryearpropertyclass[[10]] <- myresults23[[10]]$"Prior  Year Property Class"
myprioryearpropertyclass[[33]] <- myresults23[[33]]$"Property Class"
table(sapply(myprioryearpropertyclass, class))
mycurrentyearpropertyclass <- lapply(1:91, function(x) myresults23[[x]]$"Current Year Property Class")
which(sapply(mycurrentyearpropertyclass, is.null))
mycurrentyearpropertyclass[[33]] <- myresults23[[33]]$"Property Class"
table(sapply(mycurrentyearpropertyclass, class))
myprioryeartotalAV <- lapply(1:91, function(x) as.numeric(gsub(",", "", myresults23[[x]]$"Prior Year Total AV")))
which(sapply(mycurrentyearpropertyclass, is.null))
which(is.na(myprioryeartotalAV))
myprioryeartotalAV[[33]] <- as.numeric(gsub("\\$", "", gsub(",", "", myresults23[[33]]$"Total AV")))
table(sapply(myprioryeartotalAV, class))
mycurrentyeartotalAV <- lapply(1:91, function(x) as.numeric(gsub(",", "", myresults23[[x]]$"Current Year Total AV")))
which(sapply(mycurrentyeartotalAV, is.null))
which(is.na(myprioryeartotalAV))
mycurrentyeartotalAV[[33]] <- as.numeric(gsub("\\$", "", gsub(",", "", myresults23[[33]]$"WIP Total AV")))
table(sapply(mycurrentyeartotalAV, class))

[1] 33


character 
       91 

[1]  7 10 33


integer 
     91 

[1] 33

...

integer(0)


numeric 
     91 

integer(0)

integer(0)


numeric 
     91 

In [12]:
mytownship <- lapply(1:91, function(x) myresults23[[x]]$"Township Name")
which(sapply(mytownship, is.null))
mytownship[[33]] <- myresults23[[33]]$"Township"
table(sapply(mytownship, class))
mytd <- lapply(1:91, function(x) myresults23[[x]]$"Taxing District")
which(sapply(mytd, is.null))
mytd[[33]] <- myresults23[[33]]$"State District ID"
table(sapply(mytd, class))

[1] 33


character 
       91 

[1] 33


integer 
     91 

In [13]:
#Create the 2023 Global Workbook file by looping over each county and extracting the relevant attributes
combinedwkbk23 = data.frame(
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PriorPropertyClass = integer(),
  CurrentPropertyClass = integer(), 
  PriorAV = numeric(),
  CurrentAV = numeric()
)
for (i in 1:91) {
    newdata = data.frame(
      County = myresults23[[i]]$'County',
      Township = mytownship[[i]],
      TaxDistrict = mytd[[i]],
      ParcelNumber = myparcelnumbers[[i]],
      PriorPropertyClass = myprioryearpropertyclass[[i]],
      CurrentPropertyClass = mycurrentyearpropertyclass[[i]], 
      PriorAV = myprioryeartotalAV[[i]],
      CurrentAV = mycurrentyeartotalAV[[i]]
    )
    combinedwkbk23 = rbind(combinedwkbk23,newdata)
}
head(combinedwkbk23)


,County,Township,TaxDistrict,ParcelNumber,PriorPropertyClass,CurrentPropertyClass,PriorAV,CurrentAV
,<chr>,<chr>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>
1,Adams,PREBLE TOWNSHIP,12,01-01-01-100-001.000-012,101,101,216600,231500
2,Adams,PREBLE TOWNSHIP,12,01-01-01-100-002.000-012,100,100,55100,69700
3,Adams,PREBLE TOWNSHIP,12,01-01-01-100-002.500-012,100,100,72600,92000
4,Adams,PREBLE TOWNSHIP,12,01-01-01-100-004.000-012,685,685,232000,244100
5,Adams,PREBLE TOWNSHIP,12,01-01-01-100-005.000-012,610,610,0,0
6,Adams,PREBLE TOWNSHIP,12,01-01-01-100-006.000-012,610,610,0,0


## Create 2023 Global Sales Reconciliation File

In [14]:
# Create 2023 Global Sales Reconciliation File by looping over each county and extracting the relevant attributes
combinedexc23 = data.frame(
  SaleID = character(),
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PropertyClass = integer(),
  PostedAV = numeric(),
  CurrentAV = numeric(),
  SalePrice = numeric(),
  PostedSaleRatio = numeric(),
  SaleRatio = numeric(),
  PostedExplanation = character(),
  Explanation = character(),
  Reclassified = logical()
)
for (county in unique(combinedwkbk23$County)) {
    ctywkbk = combinedwkbk23[combinedwkbk23$County == county,]
    ##Note: Large Chunks of Marion, Morgan, Hancock, Lagrange Explanations blank
    # Read in county's sales reconciliation file
    if (gsub(" ", "", county) == 'Dekalb') {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/DeKalb County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Lagrange') {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/LaGrange County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Laporte') {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/LaPorte County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('St.Joseph')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/St. Joseph County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Pike') {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Pike County Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Pulaski') {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Pulaski County Sales Reconcilation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Miami') {
        excludedbook = read_excel('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Miami County Sales Reconciliation.xlsx',sheet='Invalids with explanation')  
    } else if (gsub(" ", "", county) == 'Kosciusko') {
        excludedbook = read_excel('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Kosciusko County Sales Reconciliation.xlsx',sheet='W Explanations')
        excludedbook = excludedbook[!is.na(excludedbook$Explanation),]
    } else if (gsub(" ", "", county) == 'Fountain') {
        excludedbook = read_excel('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Fountain County Sales Reconciliation.xlsx')
        colnames(excludedbook)[colnames(excludedbook) == '...12'] <- "Explanation"
    } else if (gsub(" ", "", county) == 'Clinton') {
        excludedbook = read_excel('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/Clinton County Sales Reconciliation.xlsx',sheet='invalids with explanations')
        excludedbook$Explanation[is.na(excludedbook$Explanation)] = excludedbook$'Sale Ratio'[is.na(excludedbook$Explanation)]
    } else {
        excludedbook = read_excel(paste0('tippecanoe/data/2023 Ratio Study - Sales Reconciliation Submissions/',gsub(" ", "", county),' County Sales Reconciliation.xlsx'))
    }

    #ensure that column names are standardized
    colnames(excludedbook)[colnames(excludedbook) == "FormattedParcelNumber"] <- "Formatted Parcel Number"
    colnames(excludedbook)[colnames(excludedbook) == "PropertyClass"] <- "Property Class"
    colnames(excludedbook)[colnames(excludedbook) == "TotalAV"] <- "Total AV"
    colnames(excludedbook)[colnames(excludedbook) == "SalePrice"] <- "Sale Price"
    colnames(excludedbook)[colnames(excludedbook) == "SaleRatio"] <- "Sale Ratio"
    colnames(excludedbook)[colnames(excludedbook) == "Sales Price"] <- "Sale Price"
    colnames(excludedbook)[colnames(excludedbook) == 'Formatted Parcel Number...6'] <- "Formatted Parcel Number"
    colnames(excludedbook)[colnames(excludedbook) == "TaxDistrict"] <- "Tax District"

    ## Calculate Sale Ratio if not included on county's sales reconciliation file
    if (gsub(" ", "", county) %in% c('Bartholomew','Blackford','Carroll','Dearborn','Delaware','Fayette'
                                     ,'Fulton','Gibson','Henry','Jasper','Jay','Kosciusko','Madison'
                                    ,'Miami','Ohio','Perry','Posey','Randolph','Rush','Scott','Sullivan'
                                    ,'Switzerland','Vermillion')) {
        excludedbook$'Sale Ratio' = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price')
    }

    # Warrick is missing many columns included on all other files such as Assessed Value, Property Class, etc
    # Replicate these values from the county workbook by joining by parcel number
    if (gsub(" ", "", county) %in% c('Warrick')) {
        colnames(excludedbook)[colnames(excludedbook) == "ParcelNumber"] <- "Formatted Parcel Number"
        colnames(excludedbook)[colnames(excludedbook) == "Reasoning"] <- "Explanation"
        excluded_indices <- match(excludedbook$'Formatted Parcel Number',ctywkbk$ParcelNumber)
        excludedbook$'Total AV'[!is.na(excluded_indices)] <- 
            ctywkbk$CurrentAV[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$Township[!is.na(excluded_indices)] <- 
            ctywkbk$Township[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$'Tax District'[!is.na(excluded_indices)] <- 
            ctywkbk$TaxDistrict[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$'Property Class'[!is.na(excluded_indices)] <- 
            ctywkbk$CurrentPropertyClass[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$'Property Class'[!is.na(excluded_indices)] <- 
            ctywkbk$CurrentPropertyClass[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$'Sale Ratio' = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price')
    }
    if (gsub(" ", "", county) %in% c('Laporte')) {
        excludedbook$'Sale Ratio' = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price')
    }
    if (gsub(" ", "", county) %in% c('Parke')) {
        excludedbook$'Sale Ratio' = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price')
    }
    if (gsub(" ", "", county) %in% c('Bartholomew')) {
        excludedbook$Explanation[is.na(excludedbook$Explanation)] = excludedbook$Comment[is.na(excludedbook$Explanation)]
    }

    #Store the explanations as entered by the county
    postedexp <- ifelse(is.na(excludedbook$Explanation), "Blank", excludedbook$Explanation)

    # Create new explanation if explanation entry is blank
    if (gsub(" ", "", county) %in% c('Marion','Morgan','Lagrange','Hancock')) {
        excludedbook$Explanation[is.na(excludedbook$Explanation)] = 'NO REASON PROVIDED'
    }

    #Check for missing columns
    if (!("Explanation" %in% colnames(excludedbook))) {
        message("Column 'Explanation' not found in ", county)
    }
    if (!("Formatted Parcel Number" %in% colnames(excludedbook))) {
        message("Column 'Formatted Parcel Number' not found in ", county)
    }
    if (!("Total AV" %in% colnames(excludedbook))) {
        message("Column 'Total AV' not found in ", county)
    }
    if (!("Sale Price" %in% colnames(excludedbook))) {
        message("Column 'Sale Price' not found in ", county)
    }
    if (!("Sale Ratio" %in% colnames(excludedbook))) {
        message("Column 'Sale Ratio' not found in ", county)
    }
    if (!("Property Class" %in% colnames(excludedbook))) {
        message("Column 'Property Class' not found in ", county)
    }
    if (!("SDFID" %in% colnames(excludedbook))) {
        message("Column 'SDFID' not found in ", county)
    }
    if (!("Tax District" %in% colnames(excludedbook))) {
        message("Column 'Tax District' not found in ", county)
    }

    #Use grep statements and fuzzy matching to match keywords to explanation buckets
    # classify to explanation buckets with least amount of keywords first
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("golf", Explanation, ignore.case = TRUE) ~ "GOLF COURSE",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("LEASE", Explanation, ignore.case = TRUE) ~ "LEASE",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("campground", Explanation, ignore.case = TRUE) ~ "CAMPGROUND",
        grepl("JELLYSTONE", Explanation, ignore.case = TRUE) ~ "CAMPGROUND",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("common", Explanation, ignore.case = TRUE) ~ "COMMON GROUND",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("permit", Explanation, ignore.case = TRUE) ~ "BUILDING PERMIT",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("MOBILE", Explanation, ignore.case = TRUE) ~ "MOBILE HOME",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("special use", Explanation, ignore.case = TRUE) ~ "SPECIAL USE",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("Adja", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("Adj owner", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("Adjo", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("next door", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("duplicate", Explanation, ignore.case = TRUE) ~ "DUPLICATE",
        grepl("dup sumb", Explanation, ignore.case = TRUE) ~ "DUPLICATE",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("PARTIAL", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        grepl("Percent int", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        grepl("half int", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("INVEST", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        grepl("RENT", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        grepl("INCOME", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sold as is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        grepl("as-is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        grepl("as is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("apt", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        grepl("apartment", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        grepl("condo", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        TRUE ~ Explanation  
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("OUTSIDE", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("time frame", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("future sale", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("conveyance date", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("CONSTRUCTION", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new plat", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new home", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new house", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("only used 1 year", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("prior", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("before sale", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("changes before", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("changebefore", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("CLASS", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("commercial", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("not com", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("notcom", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("type changed", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("industrial", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("LIMITED", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("INFO", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("NAME CHANGE", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("No response", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("unable to verify", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("no sale price", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sufficient", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("TIME ON MARKET", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("TIME ON THE MARKET", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("days on market", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("same day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("1 day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("one day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("trend", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("not enough", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("to study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("to do study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("lack of sales", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("not using", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("sales for study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("RELIGIO", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("PROFIT", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("GOV", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("CHARIT", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("EDUCATI", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("school", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("exempt", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("NFP", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sold for less than $", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("\\$\\w+,*\\w+\\s*SALE", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("low sale price", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("ZERO sales price", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sale amount less", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sale less th", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sold for under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("SOLD under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("and under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("in use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("PRIMARY USE", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("USE CHANGE", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("CHANGE IN", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("CHANGE UN", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("changed use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("change of use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("changeinuse", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("vac", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("v/L", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("INACTIVE", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("abandoned", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("bare", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        Explanation == "VA" ~ "VACANT LAND",
        grepl("HOMESITE", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("dwelling", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("residential structure", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("dev", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("BUILDER", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("BUSINESS", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("CONSOLID", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("REBUILD", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("llc", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("holdings company", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("lending inst", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("reo", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("corporat", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("MERGER", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("not exist", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("ACTIVE PARCEL", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("wrong parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer a parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer exists", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("retired", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("is not active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("not an active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("deleted parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("parcelisnolonger", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no loner", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("BANKRUP", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("FORECLO", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("FORCLO", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("DIVORCE", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("forced", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("distressed", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("duress", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("hardship", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("court", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("death", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("compulsory", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("repo", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("judicial", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("deceased", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AFTER SALE", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANT ph", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANT ch", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANTch", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("PHYSICAL CHANGE", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("remodel", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("reno", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("changes after", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("improve", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("add", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("updat", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("post sale", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("since last assessment", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("after the sale", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("changeafter", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("twice", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2X", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("more than", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("more then", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("three times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("resold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("re-sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2 sales", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("first of two", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("sales in one y", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("4 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("3 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("second sale", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("also sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("again", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("later date", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("same parcel sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("months later", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("times in one year", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("multiple sales", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("several times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("family", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("relat", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("LATive", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("AFFILIA", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("SAME OWNER", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("S LENGTH", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("IS TENANT", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same company", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("inherit", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same person", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same entity", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("friend", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("arms len", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("arms-length", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("ated org", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("survivorship", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("probate", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("heir", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AUCTION", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("ESTATE", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("TRUST", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("BANK", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("CASH", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("QUIT", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("QC", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("Q/C", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("TAX", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("SHORT SALE", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("CONTRACT", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("HUD", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("GUARDIANSHIP", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("sheriff", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("lc payoff", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("financ", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("trade", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("arbitration", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("FIXER", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("flip", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("CONDITION", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("COND issue", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("liveable", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("uninhabit", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("rehab", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("fire", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("remov", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("bad shape", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("demo", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("repair", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("contamination", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("condemn", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("rough shape", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("burnt", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("needs lots of", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("open market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("closed sale", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("hit the market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never hit", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never listed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("NOT listed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("NOT on", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("fsbo", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("buyer approached", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("private", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never on market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("for sale by owner", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("personal rep", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("word of mouth", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("sold by owner", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("market time", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("expos", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("advertise", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("off market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("not marketed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("out of market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("no listing", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("non marketed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never on the market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        TRUE ~ Explanation
      ))
     excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("discount", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("priced to sell", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("intangible", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("per. prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("nontangible", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("sell off", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("goodwill", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("personal prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("person. prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("seller paid", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("appraisal completed", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("value set on appeal", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("external valu", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("INCORRECT SALE PRICE", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("yard items", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("per.prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("furnish", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("bid", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("PP ", Explanation, ignore.case = FALSE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("includes pp", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("max depreciation", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("out building", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("outbuilding", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("assessed", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("less than", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("motivated", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("concessions", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("price drop", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("typical", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("below market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("indicative", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("considerably", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("outlier", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("market variab", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("reflect", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("reduced", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("under market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("neighborhood", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("represent", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("overpaid", Explanation, ignore.case = TRUE) ~ "DOES NOT REFLECT MARKET VALUE",
        grepl("asking price", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("recent sale", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("over av", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("errors in assessment", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("statistical range", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("unusual for market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("knowledge of sale", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("belowmarket", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("undermarket", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("excessive price", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("% AV", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("% of AV", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("UNKNOWLEDGEABLE", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("Study Section", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("ratio study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("in study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("validsale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("deemed valid", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("-valid sale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("formatted tab", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        Explanation == "INCLUDED" ~ "ON RATIO STUDY",
        Explanation == "Included. Sales window 12/01/22 - 11/30/23" ~ "ON RATIO STUDY",
        Explanation == "Included in Res Imp Study" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale- Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "Used" ~ "ON RATIO STUDY",
        Explanation == "used" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale-Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "VALID" ~ "ON RATIO STUDY",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("mul", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("MPS", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("MP Sale", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("parcel sale", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("package", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("combin", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("with another", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("split", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 properties", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("partition", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 parcel", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("contiguous", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("two parcel", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 part", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("two separate", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("group of", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("prices were arbit", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("4 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("bulk property", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("shares sale price", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("sold with", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("several", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("20 properties", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("29 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("30 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AG", Explanation, ignore.case = TRUE) ~ "AG LAND",
        grepl("farm", Explanation, ignore.case = TRUE) ~ "AG LAND",
        grepl("barn", Explanation, ignore.case = TRUE) ~ "AG LAND",
        TRUE ~ Explanation 
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("MIXED", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        grepl("PORTFOLIO", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        grepl("mix use", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        TRUE ~ Explanation 
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        Explanation == "INVALID" ~ "NO REASON PROVIDED",
        grepl('\u00A0',Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        Explanation == "Invalid" ~ "NO REASON PROVIDED",
        Explanation == "invalid sale - unintentionally included" ~ "NO REASON PROVIDED",
        Explanation == "invalid" ~ "NO REASON PROVIDED",
        Explanation == "NOT VALID" ~ "NO REASON PROVIDED",
        Explanation == "not valid" ~ "NO REASON PROVIDED",
        Explanation == "INVALID-" ~ "NO REASON PROVIDED",
        Explanation == "invalid-" ~ "NO REASON PROVIDED",
        Explanation == "N/A" ~ "NO REASON PROVIDED",
        Explanation == " " ~ "NO REASON PROVIDED",
        grepl("other", Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        grepl("do not use", Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        TRUE ~ Explanation
      ))

    ## Match county workbook and sales rec by parcel number
    ## Store Assessed Value from Sales Rec as PostedAV, Assessed Value from Workbook as CurrentAV
    ## Extract Township from workbook
    
        excluded_indices <- match(excludedbook$'Formatted Parcel Number',ctywkbk$ParcelNumber)
        postedAV = as.numeric(excludedbook$'Total AV')
        excludedbook$'Total AV'[!is.na(excluded_indices)] <- ctywkbk$CurrentAV[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$Township[!is.na(excluded_indices)] = ctywkbk$Township[excluded_indices[!is.na(excluded_indices)]]

    ## Create Global Sales Reconciliation file using information from each county
    ## Make sure to store both PostedAV and CurrentAV, then calcluate PostedSaleRatio and SaleRatio
    ## Make sure to include the original explanation from the sales rec file and the new explanation bucket
        newdata = data.frame(
          SaleID = excludedbook$SDFID,
          County = rep(county,nrow(excludedbook)),
          Township = excludedbook$Township,
          TaxDistrict = as.integer(excludedbook$'Tax District'),
          ParcelNumber = excludedbook$'Formatted Parcel Number',
          PropertyClass = excludedbook$'Property Class',
          PostedAV = postedAV,
          CurrentAV = as.numeric(excludedbook$'Total AV'),
          SalePrice = as.numeric(excludedbook$'Sale Price'),
          PostedSaleRatio = postedAV / as.numeric(excludedbook$'Sale Price'),
          SaleRatio = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price'),
          PostedExplanation = postedexp,
          Explanation = excludedbook$Explanation,
          Reclassified = (excludedbook$Explanation %in% c(
  "SALE DOES NOT REFLECT MARKET VALUE",
  "AV DOES NOT REFLECT MARKET VALUE",
  "NOT LISTED ON OPEN MARKET",
  "LOW SALE PRICE",
  "MULTI PARCEL SALE",
  "PRE-EXISTING RELATIONSHIP",
  "ADJACENT OWNER",
  "INVESTMENT/RENTAL",
  "CHANGES AFTER",
  "CHANGE IN USE",
  "DUPLICATE",
  "SOLD MORE THAN ONCE IN ONE YEAR",
  "AG LAND",
  "BUSINESS TRANSACTION",
  "FORCED SALE",
  "SOLD AS IS",
  "UNIQUE SALE",
  "INSUFFICIENT TIME ON MARKET",
  "VACANT LAND",
  "RELATED ASSETS",
  "SOLD TO EXEMPT ORGANIZATION",
  "NO REASON PROVIDED",
  "PARTIAL INTEREST",
  "LIMITED INFO",
  "APARTMENT",
  "GOLF COURSE",
  "INVALID PARCEL NUMBER",
  "INVALID PROPERTY CLASS",
  "LEASE",
  "CAMPGROUND",
  "COMMON GROUND",
  "BUILDING PERMIT",
  "MOBILE HOME",
  "NOT ENOUGH SALES FOR TRENDING",
  "OUTSIDE TIMEFRAME",
  "SPECIAL USE",
  "NEW CONSTRUCTION",
  "POOR CONDITION",
  "CHANGES PRIOR"
))
        )
        combinedexc23 = rbind(combinedexc23,newdata)
}
head(combinedexc23)

Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialised column: `Township`."
Warning message:
"Unknown or uninitialis

,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,PostedExplanation,Explanation,Reclassified
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<lgl>
1,C01-2021-0040330,Adams,WASHINGTON TOWNSHIP,22,01-05-03-200-010.000-022,510,77100,95800,134000,0.5753731,0.7149254,Sold Twice,SOLD MORE THAN ONCE IN ONE YEAR,TRUE
2,C01-2021-0040336,Adams,WASHINGTON TOWNSHIP,22,01-05-03-113-101.000-022,510,97400,101300,60000,1.6233333,1.6883333,Forced sale other than Foreclosure,FORCED SALE,TRUE
3,C01-2021-0040338,Adams,ROOT TOWNSHIP,14,01-02-34-306-004.000-014,510,46200,52300,17000,2.7176471,3.0764706,Significant Physical Change,CHANGES AFTER,TRUE
4,C01-2021-0040342,Adams,PREBLE TOWNSHIP,12,01-01-13-200-002.000-012,511,314900,367200,554750,0.5676431,0.6619198,Not Typical of the NBHD,AV DOES NOT REFLECT MARKET VALUE,TRUE
5,C01-2021-0040345,Adams,WASHINGTON TOWNSHIP,22,01-05-05-201-001.001-022,500,24100,231400,26000,0.9269231,8.9000000,Significant Physical Change,CHANGES AFTER,TRUE
6,C01-2021-0040351,Adams,WASHINGTON TOWNSHIP,22,01-05-03-306-074.000-022,510,60600,68900,20000,3.0300000,3.4450000,Not Listed on Market,NOT LISTED ON OPEN MARKET,TRUE


## Create 2023 Global Ratio Study File

In [15]:
##Create 2023 Global Ratio Study File by looping over each county and extracting the relevant attributes
combinedrat23 = data.frame(
  SaleID = character(),
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PropertyClass = integer(), 
  PostedAV = numeric(),
  CurrentAV = numeric(),
  SalePrice = numeric(),
  PostedSaleRatio = numeric(),
  SaleRatio = numeric(),
  MultiParcel = character(),
  Primary = character(),
  StudySection = character()
)
for (county in unique(combinedwkbk23$County)) {
    ctywkbk = combinedwkbk23[combinedwkbk23$County == county,]
    ## Load in the county's ratio study file - some have different sheet names
    filepath = paste0('tippecanoe/data/2023 Approved Ratio Studies/',toupper(county),' 2023 RATIO STUDY.xlsx')
    tryCatch({
        if (gsub(" ", "", county) %in% c('Fountain')) {
            ratiobook <- read_excel(filepath, sheet = 'FORMATTED')
        } else if (gsub(" ", "", county) %in% c('Crawford','Perry','Spencer','Vermillion')) {
            ratiobook = read_excel(paste0('tippecanoe/data/2023 Approved Ratio Studies/',toupper(county),' 2023 RATIO STUDY.xls'),sheet='Formatted')
        } else {
            ratiobook <- read_excel(filepath, sheet = 'Formatted')
        }
    }, error = function(e) {
        message("Error reading the Excel file: ",county,e$message)
        # Optionally print the structure of the file or a sample
        message("Attempting to read the first few rows.")
        test_read <- read_excel(file_path, sheet = sheets[1])  # Read the first sheet
        print(head(test_read))  # Print the first few rows
    })

    # Ensure column names are standardized
    colnames(ratiobook)[colnames(ratiobook) == "ParcelNumber"] <- "Parcel Number"
    colnames(ratiobook)[colnames(ratiobook) == "PropertyClass"] <- "Property Class"
    colnames(ratiobook)[colnames(ratiobook) == "CurrentTotalAV"] <- "Current Total AV"
    colnames(ratiobook)[colnames(ratiobook) == "StudySalePrice"] <- "Study Sale Price"
    colnames(ratiobook)[colnames(ratiobook) == "Study Sales Price"] <- "Study Sale Price"
    colnames(ratiobook)[colnames(ratiobook) == "Vendor SDFID"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "State SDFID"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "Property Class Code"] <- "Property Class"
    colnames(ratiobook)[colnames(ratiobook) == "SDFid"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "State Parcel Number"] <- "Parcel Number"
    colnames(ratiobook)[colnames(ratiobook) == "SDF ID"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "StudySection"] <- "Study Section"
    colnames(ratiobook)[colnames(ratiobook) == 'TaxDistrict'] = 'Tax District'
    colnames(ratiobook)[colnames(ratiobook) == 'Taxing District'] = 'Tax District'
    colnames(ratiobook)[colnames(ratiobook) == 'TaxingDistrict'] = 'Tax District'

    #Check to make sure all columns exist to extract
    if (!("Parcel Number" %in% colnames(ratiobook))) {
        message("Column 'Parcel Number' not found in ", county)
    }
    if (!("Property Class" %in% colnames(ratiobook))) {
        message("Column 'Property Class' not found in ", county)
    }
    if (!("Current Total AV" %in% colnames(ratiobook))) {
        message("Column 'Current Total AV' not found in ", county)
    }
    if (!("Study Sale Price" %in% colnames(ratiobook))) {
        message("Column 'Study Sale Price' not found in ", county)
    }
    if (!("SDFID" %in% colnames(ratiobook))) {
        message("Column 'SDFID' not found in ", county)
    }
    if (!("MultiParcel" %in% colnames(ratiobook))) {
        message("Column 'MultiParcel' not found in ", county)
    }
    if (!("Study Section" %in% colnames(ratiobook))) {
        message("Column 'Study Section' not found in ", county)
    }
    if (!("Tax District" %in% colnames(ratiobook))) {
        message("Column 'Tax District' not found in ", county)
    }
    # Make sure to store Assessed Value from ratio study file as PostedAV and AV from Workbook as CurrentAV
    # Vigo parcel numbers are unformatted - processing must be done differently
    if (county == "Vigo") {
        ctywkbk$ParcelNumber_1 = gsub("-","",ctywkbk$ParcelNumber)
        ctywkbk$ParcelNumber_1 = gsub("\\.","",ctywkbk$ParcelNumber_1)
        ratio_indices <- match(ratiobook$"Parcel Number",ctywkbk$ParcelNumber_1)
        postedAV = as.numeric(ratiobook$'Current Total AV')
        ratiobook$'Current Total AV'[!is.na(ratio_indices)] <- ctywkbk$CurrentAV[ratio_indices[!is.na(ratio_indices)]]
        ratiobook$township[!is.na(ratio_indices)] <- ctywkbk$Township[ratio_indices[!is.na(ratio_indices)]]
        ratiobook$'Parcel Number'[!is.na(ratio_indices)] = ctywkbk$ParcelNumber[ratio_indices[!is.na(ratio_indices)]]
    } else {
        ratio_indices <- match(ratiobook$"Parcel Number",ctywkbk$ParcelNumber)
        postedAV = as.numeric(ratiobook$'Current Total AV')
        ratiobook$'Current Total AV'[!is.na(ratio_indices)] <- ctywkbk$CurrentAV[ratio_indices[!is.na(ratio_indices)]]
        ratiobook$township[!is.na(ratio_indices)] <- ctywkbk$Township[ratio_indices[!is.na(ratio_indices)]]
    }
    # Extract relevant information from each county
    # Make sure to calculate PostedSaleRatio and SaleRatio
    newdata = data.frame(
          SaleID = ratiobook$SDFID,
          County = rep(county,nrow(ratiobook)),
          Township = ratiobook$township,
          TaxDistrict = as.integer(ratiobook$'Tax District'),
          ParcelNumber = ratiobook$'Parcel Number',
          PropertyClass = ratiobook$'Property Class', 
          PostedAV = postedAV,
          CurrentAV = as.numeric(ratiobook$'Current Total AV'),
          SalePrice = as.numeric(ratiobook$'Study Sale Price'),
          PostedSaleRatio = postedAV / as.numeric(ratiobook$'Study Sale Price'),
          SaleRatio = as.numeric(ratiobook$'Current Total AV') / as.numeric(ratiobook$'Study Sale Price'),
          MultiParcel = ratiobook$MultiParcel,
          Primary = ratiobook$Primary,
          StudySection = ratiobook$'Study Section'
    )
    combinedrat23 = rbind(combinedrat23,newdata)
}
head(combinedrat23)

Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
New names:
* `` -> `...16`
Warning message:
"Unknown or uninitialised column: `township`."
New names:
* `` -> `...16`
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
New names:
* `` -> `...16`
* `` -> `...17`
* `` -> `...18`
* `` -> `...19`
* `` -> `...20`
* `` -> `...21`
* `` ->

,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,MultiParcel,Primary,StudySection
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,C01-2020-1053465,Adams,WASHINGTON TOWNSHIP,22,01-05-02-301-031.000-022,510,75700,75700,98100,0.7716616,0.7716616,N,Y,ResImp
2,C01-2020-1067378,Adams,WASHINGTON TOWNSHIP,22,01-05-03-103-060.000-022,510,72700,72700,79800,0.9110276,0.9110276,N,Y,ResImp
3,C01-2020-1102707,Adams,WASHINGTON TOWNSHIP,22,01-05-02-102-112.000-022,510,131100,131100,158500,0.8271293,0.8271293,N,Y,ResImp
4,C01-2020-1107079,Adams,ROOT TOWNSHIP,13,01-02-35-100-004.000-013,510,208000,208000,263600,0.7890744,0.7890744,N,Y,ResImp
5,C01-2020-1135724,Adams,WABASH TOWNSHIP,18,01-11-05-202-044.000-018,510,186600,186600,194200,0.9608651,0.9608651,N,Y,ResImp
6,C01-2020-1230997,Adams,ST. MARYS TOWNSHIP,15,01-06-08-300-010.000-015,510,173300,173300,213000,0.8136150,0.8136150,N,Y,ResImp


## Create 2024 Global Sales Reconciliation File

In [16]:
#Create 2024 Global Sales Reconciliation File using same process as 2023 Global Sales Reconciliation File
combinedexc24 = data.frame(
  SaleID = character(),
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PropertyClass = integer(), 
  PostedAV = numeric(),
  CurrentAV = numeric(),
  SalePrice = numeric(),
  PostedSaleRatio = numeric(),
  SaleRatio = numeric(),
  PostedExplanation = character(),
  Explanation = character(),
  Reclassified = logical()
)
for (county in unique(combinedwkbk24$County)) {
    ctywkbk = combinedwkbk24[combinedwkbk24$County == county,]
    if (gsub(" ", "", county) %in% c('Kosciusko')) {
        next  
        ## Kosciusko 2024 Sales Reconciliation Not Available
        ## Note: Large Chunks of LaGrange, Noble, Shelby Explanations are blank
    }
    if (gsub(" ", "", county) == 'Dekalb') {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/DeKalb County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) == 'Lagrange') {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/LaGrange County Sales Reconciliation.xlsx'))
    } else if (gsub(" ", "", county) == 'Laporte') {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/LaPorte County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('Jasper','Pike')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/',gsub(" ", "", county),' County Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('St.Joseph')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/St. Joseph County Sales Reconciliation.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('Benton')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/Benton County Sales Reconciliation with notes.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('Clark')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/Clark County Sales Reconciliation Excluded Sales.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('Morgan')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/Morgan County Sales Reconciliation with explanations.xlsx'))  
    } else if (gsub(" ", "", county) %in% c('Hamilton')) {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/Hamilton County Sales Reconciliation - with explanations.xlsx'))  
    } else {
        excludedbook = read_excel(paste0('tippecanoe/data/2024 Ratio Study - Sales Reconciliation Submissions/',gsub(" ", "", county),' County Sales Reconciliation.xlsx'))
    }
    colnames(excludedbook)[colnames(excludedbook) == "FormattedParcelNumber"] <- "Formatted Parcel Number"
    colnames(excludedbook)[colnames(excludedbook) == "PropertyClass"] <- "Property Class"
    colnames(excludedbook)[colnames(excludedbook) == "TotalAV"] <- "Total AV"
    colnames(excludedbook)[colnames(excludedbook) == "SalePrice"] <- "Sale Price"
    colnames(excludedbook)[colnames(excludedbook) == "SaleRatio"] <- "Sale Ratio"
    colnames(excludedbook)[colnames(excludedbook) == "TaxDistrict"] <- "Tax District"
    colnames(excludedbook)[colnames(excludedbook) == "Taxing District"] <- "Tax District"
    colnames(excludedbook)[colnames(excludedbook) == "TaxingDistrict"] <- "Tax District"
    postedexp <- ifelse(is.na(excludedbook$Explanation), "Blank", excludedbook$Explanation)
    
    if (gsub(" ", "", county) %in% c('Noble','Lagrange','Shelby')) {
        excludedbook$Explanation[(is.na(excludedbook$Explanation))] = 'NO REASON PROVIDED'
    }
    if (gsub(" ", "", county) %in% c('Starke')) {
        colnames(excludedbook)[2] <- "SDFID"
    }
    if (!("Explanation" %in% colnames(excludedbook))) {
        message("Column 'Explanation' not found in ", county)
    }
    if (!("Formatted Parcel Number" %in% colnames(excludedbook))) {
        message("Column 'Formatted Parcel Number' not found in ", county)
    }
    if (!("Total AV" %in% colnames(excludedbook))) {
        message("Column 'Total AV' not found in ", county)
    }
    if (!("Sale Price" %in% colnames(excludedbook))) {
        message("Column 'Sale Price' not found in ", county)
    }
    if (!("Sale Ratio" %in% colnames(excludedbook))) {
        message("Column 'Sale Ratio' not found in ", county)
    }
    if (!("Property Class" %in% colnames(excludedbook))) {
        message("Column 'Property Class' not found in ", county)
    }
    if (!("SDFID" %in% colnames(excludedbook))) {
        message("Column 'SDFID' not found in ", county)
    }
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("golf", Explanation, ignore.case = TRUE) ~ "GOLF COURSE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("LEASE", Explanation, ignore.case = TRUE) ~ "LEASE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("campground", Explanation, ignore.case = TRUE) ~ "CAMPGROUND",
        grepl("JELLYSTONE", Explanation, ignore.case = TRUE) ~ "CAMPGROUND",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("common", Explanation, ignore.case = TRUE) ~ "COMMON GROUND",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("permit", Explanation, ignore.case = TRUE) ~ "BUILDING PERMIT",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("MOBILE", Explanation, ignore.case = TRUE) ~ "MOBILE HOME",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("special use", Explanation, ignore.case = TRUE) ~ "SPECIAL USE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("Adja", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("Adj owner", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("Adjo", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        grepl("next door", Explanation, ignore.case = TRUE) ~ "ADJACENT OWNER",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("duplicate", Explanation, ignore.case = TRUE) ~ "DUPLICATE",
        grepl("dup sumb", Explanation, ignore.case = TRUE) ~ "DUPLICATE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("PARTIAL", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        grepl("Percent int", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        grepl("half int", Explanation, ignore.case = TRUE) ~ "PARTIAL INTEREST",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("INVEST", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        grepl("RENT", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        grepl("INCOME", Explanation, ignore.case = TRUE) ~ "INVESTMENT/RENTAL",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sold as is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        grepl("as-is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        grepl("as is", Explanation, ignore.case = TRUE) ~ "SOLD AS IS",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("apt", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        grepl("apartment", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        grepl("condo", Explanation, ignore.case = TRUE) ~ "APARTMENT",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("OUTSIDE", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("time frame", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("future sale", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("conveyance date", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        grepl("2023 sale", Explanation, ignore.case = TRUE) ~ "OUTSIDE TIMEFRAME",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("CONSTRUCTION", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new plat", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new home", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("new house", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        grepl("only used 1 year", Explanation, ignore.case = TRUE) ~ "NEW CONSTRUCTION",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("prior", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("before sale", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("changes before", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        grepl("changebefore", Explanation, ignore.case = TRUE) ~ "CHANGES PRIOR",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("CLASS", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("commercial", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("not com", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("notcom", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("type changed", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        grepl("industrial", Explanation, ignore.case = TRUE) ~ "INVALID PROPERTY CLASS",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("LIMITED", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("INFO", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("NAME CHANGE", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("No response", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("unable to verify", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        grepl("no sale price", Explanation, ignore.case = TRUE) ~ "LIMITED INFO",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sufficient", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("TIME ON MARKET", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("TIME ON THE MARKET", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("days on market", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("same day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("1 day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        grepl("one day", Explanation, ignore.case = TRUE) ~ "INSUFFICIENT TIME ON MARKET",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("trend", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("not enough", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("to study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("to do study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("lack of sales", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("not using", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        grepl("sales for study", Explanation, ignore.case = TRUE) ~ "NOT ENOUGH SALES FOR TRENDING",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("RELIGIO", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("PROFIT", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("GOV", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("CHARIT", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("EDUCATI", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("school", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("exempt", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        grepl("NFP", Explanation, ignore.case = TRUE) ~ "SOLD TO EXEMPT ORGANIZATION",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("sold for less than $", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("\\$\\w+,*\\w+\\s*SALE", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("low sale price", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("ZERO sales price", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sale amount less", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sale less th", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("sold for under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("SOLD under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        grepl("and under", Explanation, ignore.case = TRUE) ~ "LOW SALE PRICE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("in use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("PRIMARY USE", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("USE CHANGE", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("CHANGE IN", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("CHANGE UN", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("changed use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("change of use", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        grepl("changeinuse", Explanation, ignore.case = TRUE) ~ "CHANGE IN USE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("vac", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("v/L", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("INACTIVE", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("abandoned", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("bare", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        Explanation == "VA" ~ "VACANT LAND",
        grepl("HOMESITE", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("dwelling", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        grepl("residential structure", Explanation, ignore.case = TRUE) ~ "VACANT LAND",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("dev", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("BUILDER", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("BUSINESS", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("CONSOLID", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("REBUILD", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("llc", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("holdings company", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("lending inst", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("reo", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("corporat", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        grepl("MERGER", Explanation, ignore.case = TRUE) ~ "BUSINESS TRANSACTION",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("not exist", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("ACTIVE PARCEL", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("wrong parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer a parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer exists", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("retired", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("is not active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("not an active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("deleted parcel", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("parcelisnolonger", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no longer active", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        grepl("no loner", Explanation, ignore.case = TRUE) ~ "INVALID PARCEL NUMBER",
        TRUE ~ Explanation # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("BANKRUP", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("FORECLO", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("FORCLO", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("DIVORCE", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("forced", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("distressed", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("duress", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("hardship", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("court", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("death", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("compulsory", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("repo", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("judicial", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        grepl("deceased", Explanation, ignore.case = TRUE) ~ "FORCED SALE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AFTER SALE", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANT ph", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANT ch", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("SIGNIFICANTch", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("PHYSICAL CHANGE", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("remodel", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("reno", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("changes after", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("improve", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("add", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("updat", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("post sale", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("since last assessment", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("after the sale", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        grepl("changeafter", Explanation, ignore.case = TRUE) ~ "CHANGES AFTER",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("twice", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2X", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("more than", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("more then", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("three times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("resold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("re-sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2 sales", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("sales in one y", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("first of two", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("2 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("4 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("3 times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("second sale", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("also sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("again", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("later date", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("same parcel sold", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("months later", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("times in one year", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("multiple sales", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        grepl("several times", Explanation, ignore.case = TRUE) ~ "SOLD MORE THAN ONCE IN ONE YEAR",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("family", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("relat", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("LATive", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("AFFILIA", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("SAME OWNER", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("S LENGTH", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("IS TENANT", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same company", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("inherit", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same person", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("same entity", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("friend", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("arms len", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("arms-length", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("ated org", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("survivorship", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("probate", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        grepl("heir", Explanation, ignore.case = TRUE) ~ "PRE-EXISTING RELATIONSHIP",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AUCTION", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("ESTATE", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("TRUST", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("BANK", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("CASH", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("QUIT", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("QC", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("Q/C", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("TAX", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("SHORT SALE", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("CONTRACT", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("HUD", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("GUARDIANSHIP", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("sheriff", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("lc payoff", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("financ", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("trade", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        grepl("arbitration", Explanation, ignore.case = TRUE) ~ "UNIQUE SALE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("FIXER", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("flip", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("CONDITION", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("COND issue", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("liveable", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("uninhabit", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("rehab", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("fire", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("remov", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("bad shape", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("demo", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("repair", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("contamination", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("condemn", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("rough shape", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("burnt", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        grepl("needs lots of", Explanation, ignore.case = TRUE) ~ "POOR CONDITION",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("open market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("closed sale", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("hit the market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never hit", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never listed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("NOT listed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("NOT on", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("fsbo", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("buyer approached", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("private", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never on market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("for sale by owner", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("personal rep", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("word of mouth", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("sold by owner", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("market time", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("expos", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("advertise", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("off market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("not marketed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("out of market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("no listing", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("non marketed", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        grepl("never on the market", Explanation, ignore.case = TRUE) ~ "NOT LISTED ON OPEN MARKET",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
     excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("discount", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("priced to sell", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("intangible", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("per. prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("nontangible", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("sell off", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("goodwill", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("personal prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("person. prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("seller paid", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("appraisal completed", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("value set on appeal", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("external valu", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("INCORRECT SALE PRICE", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("yard items", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("per.prop", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("furnish", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("bid", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("PP ", Explanation, ignore.case = FALSE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("includes pp", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("max depreciation", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("out building", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("outbuilding", Explanation, ignore.case = TRUE) ~ "SALE DOES NOT REFLECT MARKET VALUE",
        grepl("assessed", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("less than", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("motivated", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("concessions", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("price drop", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("typical", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("below market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("indicative", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("considerably", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("outlier", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("market variab", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("reflect", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("reduced", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("under market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("neighborhood", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("represent", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("overpaid", Explanation, ignore.case = TRUE) ~ "DOES NOT REFLECT MARKET VALUE",
        grepl("asking price", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("recent sale", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("over av", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("errors in assessment", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("statistical range", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("unusual for market", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("knowledge of sale", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("belowmarket", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("undermarket", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("excessive price", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("% AV", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("% of AV", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        grepl("UNKNOWLEDGEABLE", Explanation, ignore.case = TRUE) ~ "AV DOES NOT REFLECT MARKET VALUE",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("Study Section", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        ##grepl("included", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("ratio study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        ##grepl("used", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("in study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("validsale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("deemed valid", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("-valid sale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("formatted tab", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        Explanation == "INCLUDED" ~ "ON RATIO STUDY",
        Explanation == "Included. Sales window 12/01/22 - 11/30/23" ~ "ON RATIO STUDY",
        Explanation == "Included in Res Imp Study" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale- Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "Used" ~ "ON RATIO STUDY",
        Explanation == "used" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale-Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "VALID" ~ "ON RATIO STUDY",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("mul", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("MPS", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("MP Sale", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("parcel sale", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("package", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("combin", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("with another", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("split", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 properties", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("partition", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 parcel", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("contiguous", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("two parcel", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("2 part", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("two separate", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("group of", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("prices were arbit", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("4 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("bulk property", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("shares sale price", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("sold with", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("several", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("20 properties", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("29 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        grepl("30 parcels", Explanation, ignore.case = TRUE) ~ "MULTI PARCEL SALE",
        TRUE ~ Explanation
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("AG", Explanation, ignore.case = TRUE) ~ "AG LAND",
        grepl("farm", Explanation, ignore.case = TRUE) ~ "AG LAND",
        grepl("barn", Explanation, ignore.case = TRUE) ~ "AG LAND",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("MIXED", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        grepl("PORTFOLIO", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        grepl("mix use", Explanation, ignore.case = TRUE) ~ "RELATED ASSETS",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        grepl("Study Section", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        ##grepl("included", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("ratio study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        ##grepl("used", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("in study", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("validsale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("deemed valid", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("-valid sale", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        grepl("formatted tab", Explanation, ignore.case = TRUE) ~ "ON RATIO STUDY",
        Explanation == "INCLUDED" ~ "ON RATIO STUDY",
        Explanation == "Included. Sales window 12/01/22 - 11/30/23" ~ "ON RATIO STUDY",
        Explanation == "Included in Res Imp Study" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale- Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "Used" ~ "ON RATIO STUDY",
        Explanation == "used" ~ "ON RATIO STUDY",
        Explanation == "Valid Sale-Used in ResImp" ~ "ON RATIO STUDY",
        Explanation == "VALID" ~ "ON RATIO STUDY",
        TRUE ~ Explanation  # Keep other descriptions unchanged
      ))
    excludedbook <- excludedbook %>%
      mutate(Explanation = case_when(
        Explanation == "INVALID" ~ "NO REASON PROVIDED",
        grepl('\u00A0',Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        Explanation == "Invalid" ~ "NO REASON PROVIDED",
        Explanation == "invalid sale - unintentionally included" ~ "NO REASON PROVIDED",
        Explanation == "invalid" ~ "NO REASON PROVIDED",
        Explanation == "NOT VALID" ~ "NO REASON PROVIDED",
        Explanation == "not valid" ~ "NO REASON PROVIDED",
        Explanation == "INVALID-" ~ "NO REASON PROVIDED",
        Explanation == "invalid-" ~ "NO REASON PROVIDED",
        Explanation == "N/A" ~ "NO REASON PROVIDED",
        Explanation == " " ~ "NO REASON PROVIDED",
        grepl("other", Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        grepl("do not use", Explanation, ignore.case = TRUE) ~ "NO REASON PROVIDED",
        TRUE ~ Explanation
      ))
        excluded_indices <- match(excludedbook$'Formatted Parcel Number',ctywkbk$ParcelNumber)
        postedAV = as.numeric(excludedbook$'Total AV')
        excludedbook$'Total AV'[!is.na(excluded_indices)] <- ctywkbk$CurrentAV[excluded_indices[!is.na(excluded_indices)]]
        excludedbook$township[!is.na(excluded_indices)] = ctywkbk$Township[excluded_indices[!is.na(excluded_indices)]]
        newdata = data.frame(
          SaleID = excludedbook$SDFID,
          County = rep(county,nrow(excludedbook)),
          Township = excludedbook$township,
          TaxDistrict = as.integer(excludedbook$'Tax District'),
          ParcelNumber = excludedbook$'Formatted Parcel Number',
          PropertyClass = excludedbook$'Property Class',
          PostedAV = postedAV, 
          CurrentAV = as.numeric(excludedbook$'Total AV'),
          SalePrice = as.numeric(excludedbook$'Sale Price'),
          PostedSaleRatio = postedAV / as.numeric(excludedbook$'Sale Price'),
          SaleRatio = as.numeric(excludedbook$'Total AV') / as.numeric(excludedbook$'Sale Price'),
          PostedExplanation = postedexp,
          Explanation = excludedbook$Explanation,
          Reclassified = (excludedbook$Explanation %in% c(
  "SALE DOES NOT REFLECT MARKET VALUE",
  "AV DOES NOT REFLECT MARKET VALUE",
  "NOT LISTED ON OPEN MARKET",
  "LOW SALE PRICE",
  "MULTI PARCEL SALE",
  "PRE-EXISTING RELATIONSHIP",
  "ADJACENT OWNER",
  "INVESTMENT/RENTAL",
  "CHANGES AFTER",
  "CHANGE IN USE",
  "DUPLICATE",
  "SOLD MORE THAN ONCE IN ONE YEAR",
  "AG LAND",
  "BUSINESS TRANSACTION",
  "FORCED SALE",
  "SOLD AS IS",
  "UNIQUE SALE",
  "INSUFFICIENT TIME ON MARKET",
  "VACANT LAND",
  "RELATED ASSETS",
  "SOLD TO EXEMPT ORGANIZATION",
  "NO REASON PROVIDED",
  "PARTIAL INTEREST",
  "LIMITED INFO",
  "APARTMENT",
  "GOLF COURSE",
  "INVALID PARCEL NUMBER",
  "INVALID PROPERTY CLASS",
  "LEASE",
  "CAMPGROUND",
  "COMMON GROUND",
  "BUILDING PERMIT",
  "MOBILE HOME",
  "NOT ENOUGH SALES FOR TRENDING",
  "OUTSIDE TIMEFRAME",
  "SPECIAL USE",
  "NEW CONSTRUCTION",
  "POOR CONDITION",
  "CHANGES PRIOR"
))
        )
        combinedexc24 = rbind(combinedexc24,newdata)
}
head(combinedexc24)

Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
New names:
* `` -> `...13`
Warning messa

,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,PostedExplanation,Explanation,Reclassified
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<lgl>
1,C01-2022-0041278,Adams,WASHINGTON TOWNSHIP,22,01-05-10-203-057.000-022,510,63100,78300,120000,0.525833333,0.6525000,Private Sale,NOT LISTED ON OPEN MARKET,TRUE
2,C01-2022-0041283,Adams,WASHINGTON TOWNSHIP,22,01-05-01-102-014.089-022,500,300,237900,45000,0.006666667,5.2866667,Significant Physical Change,CHANGES AFTER,TRUE
3,C01-2022-0041284,Adams,WASHINGTON TOWNSHIP,22,01-05-03-400-025.000-022,599,14400,18100,10000,1.440000000,1.8100000,Not typical Improvement for the Neighborhood,CHANGES AFTER,TRUE
4,C01-2022-0041285,Adams,ROOT TOWNSHIP,14,01-02-35-402-026.000-014,510,84700,105300,142500,0.594385965,0.7389474,Private Sale,NOT LISTED ON OPEN MARKET,TRUE
5,C01-2022-0041290,Adams,WASHINGTON TOWNSHIP,22,01-05-03-113-087.000-022,510,99800,123300,68000,1.467647059,1.8132353,Sold Twice,SOLD MORE THAN ONCE IN ONE YEAR,TRUE
6,C01-2022-0041291,Adams,ROOT TOWNSHIP,14,01-02-35-402-028.000-014,510,74800,85600,120000,0.623333333,0.7133333,Not Listed on Market,NOT LISTED ON OPEN MARKET,TRUE


## Create 2024 Global Ratio Study File

In [17]:
## Create 2024 Global Ratio Study file using same process as 2023 Global Ratio Study File
combinedrat24 = data.frame(
  SaleID = character(),
  County = character(),
  Township = character(),
  TaxDistrict = integer(),
  ParcelNumber = character(),
  PropertyClass = integer(), 
  PostedAV = numeric(),
  CurrentAV = numeric(),
  SalePrice = numeric(),
  PostedSaleRatio = numeric(),
  SaleRatio = numeric(),
  MultiParcel = character(),
  Primary = character(),
  StudySection = character()
)
for (county in unique(combinedwkbk24$County)) {
    ctywkbk = combinedwkbk24[combinedwkbk24$County == county,]
    filepath = paste0('tippecanoe/data/2024 Approved Ratio Studies/',toupper(county),' 2024 RATIO STUDY.xlsx')
    tryCatch({
        if (gsub(" ", "", county) %in% c('Ohio')) {
            ratiobook <- read_excel(filepath, sheet='Formatted ')
        } else if (gsub(" ", "", county) %in% c('Crawford','Spencer')) {
            ratiobook = read_excel(paste0('tippecanoe/data/2024 Approved Ratio Studies/',toupper(county),' 2024 RATIO STUDY.xls'),sheet="Formatted")
        } else {
            ratiobook <- read_excel(filepath, sheet = 'Formatted')
        }
    }, error = function(e) {
        message("Error reading the Excel file: ",county,e$message)
        message("Attempting to read the first few rows.")
        test_read <- read_excel(file_path, sheet = sheets[1])
        print(head(test_read))
    })
    
    colnames(ratiobook)[colnames(ratiobook) == "ParcelNumber"] <- "Parcel Number"
    colnames(ratiobook)[colnames(ratiobook) == "PropertyClass"] <- "Property Class"
    colnames(ratiobook)[colnames(ratiobook) == "CurrentTotalAV"] <- "Current Total AV"
    colnames(ratiobook)[colnames(ratiobook) == "StudySalePrice"] <- "Study Sale Price"
    colnames(ratiobook)[colnames(ratiobook) == "State Parcel Number"] <- "Parcel Number"
    colnames(ratiobook)[colnames(ratiobook) == "State SDFID"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "SDF-ID"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "SDFid"] <- "SDFID"
    colnames(ratiobook)[colnames(ratiobook) == "Propery Class"] <- "Property Class"
    colnames(ratiobook)[colnames(ratiobook) == "Study Sales Price"] <- "Study Sale Price"
    colnames(ratiobook)[colnames(ratiobook) == "StudySection"] <- "Study Section"
    colnames(ratiobook)[colnames(ratiobook) == "TaxDistrict"] <- "Tax District"
    colnames(ratiobook)[colnames(ratiobook) == "Taxing District"] <- "Tax District"
    colnames(ratiobook)[colnames(ratiobook) == "TaxingDistrict"] <- "Tax District"
    if (!("Parcel Number" %in% colnames(ratiobook))) {
        message("Column 'Parcel Number' not found in ", county)
    }
    if (!("Property Class" %in% colnames(ratiobook))) {
        message("Column 'Property Class' not found in ", county)
    }
    if (!("Current Total AV" %in% colnames(ratiobook))) {
        message("Column 'Current Total AV' not found in ", county)
    }
    if (!("Study Sale Price" %in% colnames(ratiobook))) {
        message("Column 'Study Sale Price' not found in ", county)
    }
    if (!("SDFID" %in% colnames(ratiobook))) {
        message("Column 'SDFID' not found in ", county)
    }
    if (!("MultiParcel" %in% colnames(ratiobook))) {
        message("Column 'MultiParcel' not found in ", county)
    }
    if (!("Study Section" %in% colnames(ratiobook))) {
        message("Column 'Study Section' not found in ", county)
    }
    if (!("Tax District" %in% colnames(ratiobook))) {
        message("Column 'Tax District' not found in ", county)
    }
    ratio_indices <- match(ratiobook$"Parcel Number",ctywkbk$ParcelNumber)
    postedAV = as.numeric(ratiobook$'Current Total AV')
    ratiobook$'Current Total AV'[!is.na(ratio_indices)] <- ctywkbk$CurrentAV[ratio_indices[!is.na(ratio_indices)]]
    ratiobook$township[!is.na(ratio_indices)] = ctywkbk$Township[ratio_indices[!is.na(ratio_indices)]]
    newdata = data.frame(
      SaleID = ratiobook$SDFID,
      County = rep(county,nrow(ratiobook)),
      Township = ratiobook$township,
      TaxDistrict = as.integer(ratiobook$'Tax District'),
      ParcelNumber = ratiobook$'Parcel Number',
      PropertyClass = ratiobook$'Property Class', 
      PostedAV = postedAV,
      CurrentAV = as.numeric(ratiobook$'Current Total AV'),
      SalePrice = as.numeric(ratiobook$'Study Sale Price'),
      PostedSaleRatio = postedAV / as.numeric(ratiobook$'Study Sale Price'),
      SaleRatio = as.numeric(ratiobook$'Current Total AV') / as.numeric(ratiobook$'Study Sale Price'),
      MultiParcel = ratiobook$MultiParcel,
      Primary = ratiobook$Primary,
      StudySection = ratiobook$'Study Section'
     )
    combinedrat24 = rbind(combinedrat24,newdata)
}
head(combinedrat24)

Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
Warning message:
"Unknown or uninitialised column: `township`."
New names:
* `` -> `...16`
* `` -> `...17`
* `` -> `...18`
* `` -> `...19`
* `` -> `...20`
* `` -> `...21`
* `` -> `...22`
* `` -> `...23`
* `` -> `...24`
* `` -> `...2

,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,MultiParcel,Primary,StudySection
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,C01-2021-0040326,Adams,ROOT TOWNSHIP,13,01-02-25-100-002.000-013,511,162400,162400,154800,1.0490956,1.0490956,N,Y,ResImp
2,C01-2021-0040333,Adams,ROOT TOWNSHIP,13,01-02-21-400-009.000-013,511,243600,243600,262200,0.9290618,0.9290618,N,Y,ResImp
3,C01-2021-0040334,Adams,WASHINGTON TOWNSHIP,22,01-05-04-204-018.000-022,510,136900,136900,104500,1.3100478,1.3100478,N,Y,ResImp
4,C01-2021-0040335,Adams,WASHINGTON TOWNSHIP,22,01-05-02-101-028.000-022,510,181900,181900,172700,1.0532716,1.0532716,N,Y,ResImp
5,C01-2021-0040337,Adams,ROOT TOWNSHIP,14,01-02-36-301-022.000-014,510,263400,263400,268000,0.9828358,0.9828358,N,Y,ResImp
6,C01-2021-0040341,Adams,WASHINGTON TOWNSHIP,22,01-05-03-115-016.000-022,510,101700,101700,101600,1.0009843,1.0009843,N,Y,ResImp


## Cleaning Global Files

In [18]:
library(stringr)
df_list <- list(
  combinedwkbk23 = combinedwkbk23,
  combinedwkbk24 = combinedwkbk24,
  combinedrat24  = combinedrat24,
  combinedrat23  = combinedrat23,
  combinedexc23  = combinedexc23,
  combinedexc24  = combinedexc24
)

# Modify all township names to title case and remove 'Township'
# ex. WABASH TOWNSHIP --> Wabash
df_list <- lapply(df_list, function(df) {
  df$Township <- str_to_title(df$Township)
  df$Township <- gsub(" Township", "", df$Township)
  df
})

# Assign the updated data frames back to their original names
list2env(df_list, envir = .GlobalEnv)
head(combinedwkbk23)

<environment: R_GlobalEnv>

,County,Township,TaxDistrict,ParcelNumber,PriorPropertyClass,CurrentPropertyClass,PriorAV,CurrentAV
,<chr>,<chr>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>
1,Adams,Preble,12,01-01-01-100-001.000-012,101,101,216600,231500
2,Adams,Preble,12,01-01-01-100-002.000-012,100,100,55100,69700
3,Adams,Preble,12,01-01-01-100-002.500-012,100,100,72600,92000
4,Adams,Preble,12,01-01-01-100-004.000-012,685,685,232000,244100
5,Adams,Preble,12,01-01-01-100-005.000-012,610,610,0,0
6,Adams,Preble,12,01-01-01-100-006.000-012,610,610,0,0


In [19]:
# Remove all rows with NA in any column
df_list <- lapply(df_list, function(df) {
  df <- df[complete.cases(df), ]
  df
})

list2env(df_list, envir = .GlobalEnv)


<environment: R_GlobalEnv>

In [20]:
# Combine Multi Parcel sales into a single row describing that sale instead of
# a row for each parcel in the transaction
mps = combinedrat23[combinedrat23$MultiParcel == 'Y',] %>%
  mutate(
    # Extract final (numeric) part of SaleID
    numeric_suffix = as.integer(gsub(".*-(\\d+)$", "\\1", SaleID)),
    # Extract SaleID year
    base_saleid = gsub("-(\\d+)$", "", SaleID)
  ) %>%
  # Group by SaleID and show the frequency of each SaleID to identify duplicates
  group_by(SaleID) %>%
  mutate(freq = n()) %>%
  ungroup() %>%
  # Sort by year, and within year sort by numeric part of SaleID
  arrange(base_saleid, numeric_suffix) %>%
  # For rows with a unique SaleID, see if there is a sequential SaleID to create a group; otherwise, 
  # the sale is not part of a group
  group_by(base_saleid) %>%
  mutate(
    seq_group = if_else(freq == 1, cumsum(c(0, diff(numeric_suffix) != 1)) + 1, NA_integer_)
  ) %>%
  ungroup() %>%
  # For rows with duplicate SaleID, all sales with that SaleID are part of a group
  mutate(
    group_id = if_else(freq == 1,
                       paste0(base_saleid, "-", seq_group),
                       SaleID)
  ) %>%
  group_by(group_id) %>%
    filter(!is.na(County)) %>%
   # Extract parcel-identifying information for the primary parcel only
   # Extract AV, Sale Price, and SaleRatio information by summing the AV of all parcels in the transaction,
   # summing only the unique sale prices (all sales in a group may be reported with a single overall price),
   # and then dividing the summed AV by the summed Sale Price
    summarise(SaleID = first(unique(SaleID)), 
              County = ifelse(first(County[Primary == 'Y']) == 'NA',first(na.omit(County)),
                                     first(County[Primary == 'Y'])), 
              Township = ifelse(first(Township[Primary == 'Y']) == 'NA',first(na.omit(Township)),
                                first(Township[Primary == 'Y'])),
              TaxDistrict = ifelse(first(TaxDistrict[Primary == 'Y']) == 'NA',first(na.omit(TaxDistrict)),
                                first(TaxDistrict[Primary == 'Y'])),
              ParcelNumber = first(unique(ParcelNumber[grepl("-", ParcelNumber)])), 
              PropertyClass = ifelse(first(PropertyClass[Primary == 'Y']) == 'NA',first(na.omit(PropertyClass)),
                                     first(PropertyClass[Primary == 'Y'])),
              PostedAV = sum(PostedAV), CurrentAV = sum(CurrentAV), SalePrice = sum(unique(SalePrice)),
              PostedSaleRatio = sum(PostedAV) / sum(unique(SalePrice)), SaleRatio = sum(CurrentAV) / sum(unique(SalePrice)), 
              MultiParcel = 'Y', Primary = 'Y',StudySection = ifelse(first(StudySection[Primary == 'Y']) == 'NA',
                                                                     first(na.omit(StudySection)),
                                     first(StudySection[Primary == 'Y'])), Count = n()
              
  )
# Add these combined Multi Parcel Sales back to the original data frame
head(mps)
truemps = mps[mps$Count > 1,]
falsemps = mps[mps$Count < 2,]
falsemps$MultiParcel = 'N'
combinedrat23 = combinedrat23[combinedrat23$MultiParcel == 'N',]
truemps = truemps[,2:15]
falsemps = falsemps[,2:15]
combinedrat23 = rbind(combinedrat23,truemps,falsemps)

group_id,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,MultiParcel,Primary,StudySection,Count
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>
C01-2020-1381298,C01-2020-1381298,Adams,Root,14,01-02-34-306-008.000-014,420,70800,70800,64000,1.1062500,1.1062500,Y,Y,ComImp,2
C01-2020-1535490,C01-2020-1535490,Adams,Preble,12,01-01-35-400-019.000-012,511,120400,120400,104600,1.1510516,1.1510516,Y,Y,ResImp,2
C01-2020-5204098,C01-2020-5204098,Adams,Washington,22,01-05-03-100-001.000-022,420,94400,94400,118900,0.7939445,0.7939445,Y,Y,ComImp,3
C01-2020-7857820,C01-2020-7857820,Adams,Washington,22,01-05-04-301-030.000-022,510,189800,189800,220200,0.8619437,0.8619437,Y,Y,ResImp,2
C01-2020-9181447,C01-2020-9181447,Adams,Washington,21,01-05-11-300-022.000-021,510,147000,147000,120700,1.2178956,1.2178956,Y,Y,ResImp,2
C01-2021-0040339,C01-2021-0040339,Adams,Washington,22,01-05-03-413-015.000-022,500,11400,11400,15500,0.7354839,0.7354839,Y,Y,ResVac,2


In [21]:
# Combine Multi Parcel sales into a single row describing that sale instead of
# a row for each parcel in the transaction
mps = combinedrat24[combinedrat24$MultiParcel == 'Y',] %>%
  mutate(
    # Extract final (numeric) part of SaleID
    numeric_suffix = as.integer(gsub(".*-(\\d+)$", "\\1", SaleID)),
    # Extract SaleID year
    base_saleid = gsub("-(\\d+)$", "", SaleID)
  ) %>%
  # Group by SaleID and show the frequency of each SaleID to identify duplicates
  group_by(SaleID) %>%
  mutate(freq = n()) %>%
  ungroup() %>%
  # Sort by year, and within year sort by numeric part of SaleID
  arrange(base_saleid, numeric_suffix) %>%
  # For rows with a unique SaleID, see if there is a sequential SaleID to create a group; otherwise, 
  # the sale is not part of a group
  group_by(base_saleid) %>%
  mutate(
    seq_group = if_else(freq == 1, cumsum(c(0, diff(numeric_suffix) != 1)) + 1, NA_integer_)
  ) %>%
  ungroup() %>%
  # For rows with duplicate SaleID, all sales with that SaleID are part of a group
  mutate(
    group_id = if_else(freq == 1,
                       paste0(base_saleid, "-", seq_group),
                       SaleID)
  ) %>%
  group_by(group_id) %>%
    filter(!is.na(County)) %>%
    # Extract parcel-identifying information for the primary parcel only
    # Extract AV, Sale Price, and SaleRatio information by summing the AV of all parcels in the transaction,
    # summing only the unique sale prices (all sales in a group may be reported with a single overall price),
    # and then dividing the summed AV by the summed Sale Price
    summarise(SaleID = first(unique(SaleID)), 
              County = ifelse(first(County[Primary == 'Y']) == 'NA',first(na.omit(County)),
                                     first(County[Primary == 'Y'])), 
              Township = ifelse(first(Township[Primary == 'Y']) == 'NA',first(na.omit(Township)),
                                first(Township[Primary == 'Y'])),
              TaxDistrict = ifelse(first(TaxDistrict[Primary == 'Y']) == 'NA',first(na.omit(TaxDistrict)),
                                first(TaxDistrict[Primary == 'Y'])),
              ParcelNumber = first(unique(ParcelNumber[grepl("-", ParcelNumber)])), 
              PropertyClass = ifelse(first(PropertyClass[Primary == 'Y']) == 'NA',first(na.omit(PropertyClass)),
                                     first(PropertyClass[Primary == 'Y'])),
              PostedAV = sum(PostedAV), CurrentAV = sum(CurrentAV), SalePrice = sum(unique(SalePrice)),
              PostedSaleRatio = sum(PostedAV) / sum(unique(SalePrice)), SaleRatio = sum(CurrentAV) / sum(unique(SalePrice)), 
              MultiParcel = 'Y', Primary = 'Y',StudySection = ifelse(first(StudySection[Primary == 'Y']) == 'NA',
                                                                     first(na.omit(StudySection)),
                                     first(StudySection[Primary == 'Y'])),Count = n()
              
  )
head(mps)
truemps = mps[mps$Count > 1,]
falsemps = mps[mps$Count < 2,]
falsemps$MultiParcel = 'N'
combinedrat24 = combinedrat24[combinedrat24$MultiParcel == 'N',]
truemps = truemps[,2:15]
falsemps = falsemps[,2:15]
combinedrat24 = rbind(combinedrat24,truemps,falsemps)

group_id,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,MultiParcel,Primary,StudySection,Count
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>
C01-2021-0040347,C01-2021-0040347,Adams,Monroe,10,01-08-33-301-151.000-010,510,108600,108600,98800,1.0991903,1.0991903,Y,Y,ResImp,2
C01-2021-0040494,C01-2021-0040494,Adams,Root,14,01-02-33-200-217.000-014,510,554200,554200,430000,1.2888372,1.2888372,Y,Y,ResImp,2
C01-2021-0041074,C01-2021-0041074,Adams,Monroe,10,01-08-33-301-225.000-010,520,111500,111500,95500,1.1675393,1.1675393,Y,Y,ResImp,2
C01-2022-0041481,C01-2022-0041481,Adams,Washington,22,01-05-03-103-029.000-022,510,80900,80900,93800,0.8624733,0.8624733,Y,Y,ResImp,2
C01-2022-0041703,C01-2022-0041703,Adams,Monroe,8,01-08-10-200-002.000-008,511,317700,317700,335800,0.9460989,0.9460989,Y,Y,ResImp,2
C01-2022-0041747,C01-2022-0041747,Adams,Washington,22,01-05-03-300-037.505-022,481,683000,683000,702900,0.9716887,0.9716887,Y,Y,ComImp,3


In [22]:
# Combine Multi Parcel sales into a single row describing that sale instead of
# a row for each parcel in the transaction
# Done using the same procedure as the previous examples
mps = combinedexc23[combinedexc23$Explanation == 'MULTI PARCEL SALE',] %>%
  mutate(
    numeric_suffix = as.integer(gsub(".*-(\\d+)$", "\\1", SaleID)),
    base_saleid = gsub("-(\\d+)$", "", SaleID)
  ) %>%
  group_by(SaleID) %>%
  mutate(freq = n()) %>%
  ungroup() %>%
  arrange(base_saleid, numeric_suffix) %>%
  group_by(base_saleid) %>%
  mutate(
    seq_group = if_else(freq == 1, cumsum(c(0, diff(numeric_suffix) != 1)) + 1, NA_integer_)
  ) %>%
  ungroup() %>%
  mutate(
    group_id = if_else(freq == 1,
                       paste0(base_saleid, "-", seq_group),
                       SaleID)
  ) %>%
  group_by(group_id) %>%
    filter(!is.na(County)) %>%
    summarise(SaleID = first(unique(SaleID)), 
              County = first(na.omit(County)), 
              Township = first(na.omit(Township)),
              TaxDistrict = first(na.omit(TaxDistrict)),
              ParcelNumber = first(unique(ParcelNumber[grepl("-", ParcelNumber)])), 
              PropertyClass = first(na.omit(PropertyClass)),
              PostedAV = sum(PostedAV), CurrentAV = sum(CurrentAV), SalePrice = sum(unique(SalePrice)),
              PostedSaleRatio = sum(PostedAV) / sum(unique(SalePrice)), SaleRatio = sum(CurrentAV) / sum(unique(SalePrice)), 
              PostedExplanation = first(unique(PostedExplanation)),Explanation = 'MULTI PARCEL SALE',
              Reclassified = TRUE,Count = n()
              
  )
# Remove sales with ratios outside 0.5 - 2. Some AV and SalePrice information was mis-recorded
mps = mps[mps$SaleRatio > 0.5 & mps$SaleRatio < 2,]
head(mps)
mps = mps[,2:15]
combinedexc23 = combinedexc23[combinedexc23$Explanation != 'MULTI PARCEL SALE',]
combinedexc23 = rbind(combinedexc23,mps)

group_id,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,PostedExplanation,Explanation,Reclassified,Count
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<lgl>,<int>
C01-2022-3,C01-2022-0041537,Adams,Washington,22,01-05-04-300-015.000-022,330,664500,652800,885518,0.7504082,0.7371956,Sold with an Ag parcel,MULTI PARCEL SALE,TRUE,1
C02-2022-1,C02-2022-0180984,Allen,Wayne,74,02-13-18-254-020.000-074,510,38500,52600,49000,0.7857143,1.0734694,Package Sale,MULTI PARCEL SALE,TRUE,1
C02-2022-10,C02-2022-0181912,Allen,Wayne,74,02-12-11-429-001.000-074,480,567400,753900,1145841,0.4951821,0.6579447,Package Sale,MULTI PARCEL SALE,TRUE,3
C02-2022-106,C02-2022-0188559,Allen,Wayne,74,02-13-07-253-007.000-074,510,95500,275900,379830,0.2514283,0.7263776,Package Sale,MULTI PARCEL SALE,TRUE,2
C02-2022-107,C02-2022-0188562,Allen,Perry,58,02-02-04-351-007.000-058,510,0,217700,349830,0.0000000,0.6223023,Package Sale,MULTI PARCEL SALE,TRUE,1
C02-2022-114,C02-2022-0188704,Allen,Wayne,74,02-12-11-153-001.000-074,510,69300,102600,100000,0.6930000,1.0260000,Package Sale,MULTI PARCEL SALE,TRUE,1


In [23]:
# Combine Multi Parcel sales into a single row describing that sale instead of
# a row for each parcel in the transaction
# Done using the same procedure as the previous examples
mps = combinedexc24[combinedexc24$Explanation == 'MULTI PARCEL SALE',] %>%
  mutate(
    numeric_suffix = as.integer(gsub(".*-(\\d+)$", "\\1", SaleID)),
    base_saleid = gsub("-(\\d+)$", "", SaleID)
  ) %>%
  group_by(SaleID) %>%
  mutate(freq = n()) %>%
  ungroup() %>%
  arrange(base_saleid, numeric_suffix) %>%
  group_by(base_saleid) %>%
  mutate(
    seq_group = if_else(freq == 1, cumsum(c(0, diff(numeric_suffix) != 1)) + 1, NA_integer_)
  ) %>%
  ungroup() %>%
  mutate(
    group_id = if_else(freq == 1,
                       paste0(base_saleid, "-", seq_group),
                       SaleID)
  ) %>%
  group_by(group_id) %>%
    filter(!is.na(County)) %>%
    summarise(SaleID = first(unique(SaleID)), 
              County = first(na.omit(County)), 
              Township = first(na.omit(Township)),
              TaxDistrict = first(na.omit(TaxDistrict)),
              ParcelNumber = first(unique(ParcelNumber[grepl("-", ParcelNumber)])), 
              PropertyClass = first(na.omit(PropertyClass)),
              PostedAV = sum(PostedAV), CurrentAV = sum(CurrentAV), SalePrice = sum(unique(SalePrice)),
              PostedSaleRatio = sum(PostedAV) / sum(unique(SalePrice)), SaleRatio = sum(CurrentAV) / sum(unique(SalePrice)), 
              PostedExplanation = first(unique(PostedExplanation)),Explanation = 'MULTI PARCEL SALE',
              Reclassified = TRUE,Count = n()
              
  )

# Remove sales with ratios outside 0.5 - 2. Some AV and SalePrice information was mis-recorded
mps = mps[mps$SaleRatio > 0.5 & mps$SaleRatio < 2,]
head(mps)
mpsc = mps[,2:15]
combinedexc24 = combinedexc24[combinedexc24$Explanation != 'MULTI PARCEL SALE',]
combinedexc24 = rbind(combinedexc24,mpsc)

group_id,SaleID,County,Township,TaxDistrict,ParcelNumber,PropertyClass,PostedAV,CurrentAV,SalePrice,PostedSaleRatio,SaleRatio,PostedExplanation,Explanation,Reclassified,Count
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<lgl>,<int>
C01-2022-2,C01-2022-0041449,Adams,French,4,01-07-34-200-001.000-004,511,104700,337100,350000,0.2991429,0.9631429,Sold with an Ag parcel,MULTI PARCEL SALE,TRUE,1
C01-2022-3,C01-2022-0041537,Adams,Washington,22,01-05-04-300-015.000-022,330,664500,620700,885518,0.7504082,0.7009457,Sold with an Ag parcel,MULTI PARCEL SALE,TRUE,1
C01-2022-4,C01-2022-0041907,Adams,Monroe,8,01-08-02-300-005.000-008,511,126900,188900,300000,0.4230000,0.6296667,Sold with an Ag parcel,MULTI PARCEL SALE,TRUE,1
C02-2023-100,C02-2023-0199625,Allen,Wayne,74,02-07-35-152-010.000-074,510,214100,247800,220000,0.9731818,1.1263636,Package/Portfolio Sale,MULTI PARCEL SALE,TRUE,4
C02-2023-102,C02-2023-0199789,Allen,Wayne,74,02-12-24-281-037.000-074,510,458800,642300,480124,0.9555865,1.3377794,Package/Portfolio Sale,MULTI PARCEL SALE,TRUE,8
C02-2023-103,C02-2023-0199820,Allen,Wayne,74,02-13-07-210-001.000-074,510,83300,106400,65000,1.2815385,1.6369231,Package/Portfolio Sale,MULTI PARCEL SALE,TRUE,2


In [24]:
# Some sales were excluded and classified as 'VACANT LAND'
# Vacant land sales should be included and placed in the vacant land portion of the study
# Rewrite property classes to ensure that these properties are classified as vacant
combinedexc23$PropertyClass <- as.numeric(as.character(combinedexc23$PropertyClass))
idx <- !is.na(combinedexc23$Explanation) & (combinedexc23$Explanation == 'VACANT LAND')
combinedexc23$PropertyClass[idx] <- floor(combinedexc23$PropertyClass[idx] / 100) * 100
combinedexc24$PropertyClass <- as.numeric(as.character(combinedexc24$PropertyClass))
idx <- !is.na(combinedexc24$Explanation) & (combinedexc24$Explanation == 'VACANT LAND')
combinedexc24$PropertyClass[idx] <- floor(combinedexc24$PropertyClass[idx] / 100) * 100

In [25]:
# Separate Sales that were included and sales that were excluded
ratiostudies24 = combinedexc24[combinedexc24$Explanation == "ON RATIO STUDY",]
combinedexc24 = combinedexc24[combinedexc24$Explanation != "ON RATIO STUDY",]
nrow(ratiostudies24)
nrow(combinedexc24)
ratiostudies23 = combinedexc23[combinedexc23$Explanation == "ON RATIO STUDY",]
combinedexc23 = combinedexc23[combinedexc23$Explanation != 'ON RATIO STUDY',]
nrow(ratiostudies23)
nrow(combinedexc23)

[1] 17383

[1] 41008

[1] 19900

[1] 93229

In [42]:
# Make sure rows are not NA and save the sales that were included
ratiostudies23 <- ratiostudies23[!apply(is.na(ratiostudies23), 1, all), ]
ratiostudies24 <- ratiostudies24[!apply(is.na(ratiostudies24), 1, all), ]
write.csv(ratiostudies23, 'tippecanoe/Added_Sales_2023.csv',row.names=FALSE)
write.csv(ratiostudies24, 'tippecanoe/Added_Sales_2024.csv',row.names=FALSE)

In [26]:
nrow(combinedrat23)
nrow(combinedexc23)

[1] 116812

[1] 93229

In [45]:
# Make sure rows are not NA and save the files
nrow(combinedwkbk23)
combinedwkbk23 <- combinedwkbk23[!apply(is.na(combinedwkbk23), 1, all), ]
combinedwkbk24 <- combinedwkbk24[!apply(is.na(combinedwkbk24), 1, all), ]
combinedexc23 <- combinedexc23[!apply(is.na(combinedexc23), 1, all), ]
combinedexc24 <- combinedexc24[!apply(is.na(combinedexc24), 1, all), ]
combinedrat23 <- combinedrat23[!apply(is.na(combinedrat23), 1, all), ]
combinedrat24 <- combinedrat24[!apply(is.na(combinedrat24), 1, all), ]
combinedwkbk23 = combinedwkbk23[!duplicated(combinedwkbk23$ParcelNumber), ]
nrow(combinedwkbk23)
nrow(combinedwkbk24)
combinedwkbk24 = combinedwkbk24[!duplicated(combinedwkbk24$ParcelNumber), ]
nrow(combinedwkbk24)
write.csv(combinedwkbk23, 'tippecanoe/Combined_Workbook_2023.csv',row.names=FALSE)
write.csv(combinedwkbk24, 'tippecanoe/Combined_Workbook_2024.csv',row.names=FALSE)
write.csv(combinedexc23, 'tippecanoe/Combined_Reconciliation_2023.csv',row.names=FALSE)
write.csv(combinedexc24, 'tippecanoe/Combined_Reconciliation_2024.csv',row.names=FALSE)
write.csv(combinedrat23, 'tippecanoe/Combined_Study_2023.csv',row.names=FALSE)
write.csv(combinedrat24, 'tippecanoe/Combined_Study_2024.csv',row.names=FALSE)

[1] 4584011

[1] 3554776

[1] 2331642

[1] 2331640